# Determining the optiomal number of hidden layers and neurons for an Artificial Neural Network (ANN)

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping
import pickle

In [3]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("Active GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


CUDA Available: True
Active GPU: NVIDIA GeForce GTX 1650


In [4]:
data = pd.read_csv('Dataset/Churn_Modelling.csv')
data=data.drop(['RowNumber', 'CustomerId', 'Surname'],axis=1)

label_encoder = LabelEncoder()
data['Gender']=label_encoder.fit_transform(data['Gender'])

one_hot_encode = OneHotEncoder()

geo_encoder=one_hot_encode.fit_transform(data[['Geography']]).toarray()

geo_encoded_df=pd.DataFrame(geo_encoder ,columns=one_hot_encode.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

x = data.drop('Exited',axis=1)

y = data['Exited']


x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2, random_state=42)

scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(one_hot_encode, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [5]:
## Define a function to create the model and try different parameters(KerasClassifier)

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(x_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model



In [6]:
## Create a Keras classifier
model=KerasClassifier(layers=1,neurons=32,build_fn=create_model,verbose=1)

In [7]:

# Define the grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [50, 100]
}

In [9]:
# Perform grid search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3,verbose=1)
grid_result = grid.fit(x_train, y_train)

# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits


d:\Courses\GenAI Udemy Advanced\Customer Churn Prediction Project Using ANN\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


Epoch 1/50
250/250 [==============================] - 2s 3ms/step - loss: 0.5339 - accuracy: 0.7364
Epoch 2/50
250/250 [==============================] - 1s 3ms/step - loss: 0.4255 - accuracy: 0.8164
Epoch 3/50
250/250 [==============================] - 1s 2ms/step - loss: 0.4081 - accuracy: 0.8260
Epoch 4/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3926 - accuracy: 0.8366
Epoch 5/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3791 - accuracy: 0.8435
Epoch 6/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3676 - accuracy: 0.8494
Epoch 7/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3595 - accuracy: 0.8533
Epoch 8/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3540 - accuracy: 0.8562
Epoch 9/50
250/250 [==============================] - 1s 3ms/step - loss: 0.3492 - accuracy: 0.8571
Epoch 10/50
250/250 [==============================] - 1s 2ms/step - loss: 0.3468 - accuracy: 0.8595